# S0 — 03.01: Logistic Regression — From Features to Probabilities

**Module:** 03 — Supervised Classification  
**Prerequisites:** 00.06, 00.08, 01.01–01.06, 02.01–02.08  
**Difficulty:** introductory  
**Estimated time:** 50–70 minutes core

> **Learner question:** How can a model turn a linear feature score into a probability for a binary class, and how do we turn that probability into a class prediction?

Module 02 predicted continuous numbers. Module 03 starts classification: predicting a discrete class label. In binary classification, the target takes values `0` or `1`.

This lesson evaluates a fixed logistic-regression model. We supply `b` and `w`, compute probabilities, apply a threshold, and compare predicted classes with true labels. We do **not** train the model yet. Lesson 03.02 introduces the classification loss and gradient-descent procedure that choose coefficients automatically.

The example is synthetic, deterministic, and dimensionless. Its accuracy is arithmetic on this toy table, not evidence about real-world performance or generalization.

## S1 — Classification, binary labels, and the historical name

In **regression**, the target is a continuous value: a price, temperature, score, or measurement. In **classification**, the target is a discrete label: class `0` or class `1`, spam or not spam, approved or rejected.

Despite the word **regression**, logistic regression is a classification model: it outputs probabilities for a discrete class label. The name is historical. The model is a ‘regression’ in the sense that it fits a smooth function of the features — specifically, a linear function of the log-odds — but its job here is classification.

| Module 02 linear regression | Lesson 03.01 logistic regression |
|---|---|
| target can be any continuous number | target is `0` or `1` |
| output is a predicted value | output first is a probability |
| no threshold is needed | a threshold turns probability into class label |
| training objective already studied | classification training objective comes in 03.02 |

## S2 — A tiny synthetic binary-classification table

We will use eight synthetic, dimensionless examples. The single feature is called `x`. The target `y` is a binary class label. Most low `x` values are class `0`, and most high `x` values are class `1`. The two middle points are deliberately noisy: `x = 1` has label `1`, while `x = 2` has label `0`. That prevents the toy example from pretending that classification is always perfect.

In [1]:
import math
import numpy as np

RTOL = 0.0
ATOL = 1e-12
np.set_printoptions(precision=6, suppress=True)

X = np.array([[-2.0], [-1.0], [0.0], [1.0], [2.0], [3.0], [4.0], [5.0]], dtype=np.float64)
y = np.array([0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0], dtype=np.float64)

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("X dtype:", X.dtype)
print("y dtype:", y.dtype)

X.shape: (8, 1)
y.shape: (8,)
X dtype: float64
y dtype: float64


The data table is:

| example `i` | feature `x_i` | target `y_i` |
|---:|---:|---:|
| 1 | -2 | 0 |
| 2 | -1 | 0 |
| 3 | 0 | 0 |
| 4 | 1 | 1 |
| 5 | 2 | 0 |
| 6 | 3 | 1 |
| 7 | 4 | 1 |
| 8 | 5 | 1 |

Notation: `z_i = b + wᵀx_i` is the linear score, `p_i = σ(z_i)` is the predicted probability of class `1`, and `y_hat_i` is the hard predicted class after thresholding. Watch the notation collision: `p` can mean number of features, while `p_i` means predicted probability.

## S3 — A binary target needs a probability

Recall from probability foundations: a Bernoulli random variable takes values `{0, 1}` with `P(Y = 1) = π`. Logistic regression makes the probability of class `1` depend on the features.

For one example, the model has this shape:

```text
feature x_i → linear score z_i → probability p_i
```

For many features, logistic regression uses `z_i = b + wᵀx_i` and `p_i = σ(z_i)`. In this one-feature lesson, `z_i = b + w*x_i`.

A natural first instinct is to reuse MSE from regression and square the difference between the predicted probability and the `0`/`1` label. For this model that is a poor default: the resulting error surface is non-convex and it saturates, so the learning signal can become too small when the sigmoid saturates. Classification instead uses a purpose-built loss, introduced in 03.02.

## S4 — The sigmoid squashes a linear score into a probability

The linear score `z` can be any real number. A probability must be between `0` and `1`. Logistic regression uses the **sigmoid** function:

```text
σ(z) = 1 / (1 + exp(-z))
```

Then `p_i = σ(z_i)`. Logistic regression is the composition: features → linear score → sigmoid → probability.

In [2]:
b = -1.5
w = np.array([1.0], dtype=np.float64)

print("b:", b)
print("w:", w)

b: -1.5
w: [1.]


> **Where did these coefficients come from?** The coefficients `b = -1.5` and `w = 1.0` are hand-picked round values, chosen only so the arithmetic stays clean and the boundary lands between the two class regions. They are not the output of any training procedure. Lesson 03.02 shows how a purpose-built loss function and gradient descent find coefficients automatically.

With these values, `z_i = -1.5 + 1.0*x_i`.

Five sigmoid facts:

| Property | Meaning |
|---|---|
| `σ(0) = 0.5` | a zero score maps to probability one-half |
| `σ(z) → 1` as `z → +∞` | large positive scores approach class-1 probability `1` |
| `σ(z) → 0` as `z → -∞` | large negative scores approach class-1 probability `0` |
| `σ(-z) = 1 - σ(z)` | the curve is symmetric around `(0, 0.5)` |
| `σ` is strictly increasing | higher scores mean higher predicted probabilities |

The sigmoid is not a step function. It changes smoothly, so the model can say ‘class 1 with probability `0.62`’ instead of only ‘class 1’.

**Optional logit panel.** The inverse of the sigmoid is `logit(p) = ln(p / (1 - p))`. Therefore logistic regression has `ln(p / (1 - p)) = b + wᵀx`: the log-odds are linear in the features. For example, at `z = -0.5`, `p/(1-p) = 0.606530659712633 = exp(-0.5)`. Odds equal `e^z`, so log-odds equal `z`.

## S5 — Hand calculation: score → probability → class

The rule is:

```text
z_i = -1.5 + 1.0*x_i
p_i = σ(z_i) = 1 / (1 + exp(-z_i))
y_hat_i = 1 if p_i >= 0.5 else 0
```

The threshold is `0.5`, and the tie-break rule is: if `p_i == 0.5`, predict class `1`.

Careful with the double negative: for negative `z`, `exp(-z) = exp(positive number)`. Row 4 has `z = -0.5`, so `exp(-z) = exp(0.5) = 1.648721` — not `exp(-0.5) = 0.606531`.

### Worked rows before the complete table

The complete table below contains every row. These four rows show the main arithmetic patterns.

**Row 1: `x = -2`, `y = 0`**

```text
z = -1.5 + 1.0*(-2) = -3.5
exp(-z) = exp(3.5) = 33.115452
1 + exp(-z) = 34.115452
σ(z) = 1 / 34.115452 = 0.029312
y_hat = 0 because 0.029312 < 0.5
```

The true label is `0`, so row 1 is correct.

**Row 4: `x = 1`, `y = 1`**

```text
z = -1.5 + 1.0*(1) = -0.5
exp(-z) = exp(0.5) = 1.648721
1 + exp(-z) = 2.648721
σ(z) = 1 / 2.648721 = 0.377541
y_hat = 0 because 0.377541 < 0.5
```

The true label is `1`, so row 4 is incorrect. The probability is still informative: the model assigns class `1` probability about `0.38`, then thresholding turns that into hard class `0`.

**Row 5: `x = 2`, `y = 0`**

```text
z = -1.5 + 1.0*(2) = 0.5
exp(-z) = exp(-0.5) = 0.606531
1 + exp(-z) = 1.606531
σ(z) = 1 / 1.606531 = 0.622459
y_hat = 1 because 0.622459 >= 0.5
```

The true label is `0`, so row 5 is incorrect. This row is the anchor for keeping four quantities separate:

```text
x = 2 → z = 0.5 → p = 0.622459 → y_hat = 1, but y = 0
```

**Row 8: `x = 5`, `y = 1`**

```text
z = -1.5 + 1.0*(5) = 3.5
exp(-z) = exp(-3.5) = 0.030197
1 + exp(-z) = 1.030197
σ(z) = 1 / 1.030197 = 0.970688
y_hat = 1 because 0.970688 >= 0.5
```

The true label is `1`, so row 8 is correct.


Complete hand trace:

| i | `x_i` | `y_i` | `z_i` | `exp(-z_i)` | `1+exp(-z_i)` | `σ(z_i)` | `y_hat_i` | Correct? |
|---:|---:|---:|---:|---:|---:|---:|---:|:---:|
| 1 | -2 | 0 | -3.5 | 33.115452 | 34.115452 | 0.029312 | 0 | yes |
| 2 | -1 | 0 | -2.5 | 12.182494 | 13.182494 | 0.075858 | 0 | yes |
| 3 | 0 | 0 | -1.5 | 4.481689 | 5.481689 | 0.182426 | 0 | yes |
| 4 | 1 | 1 | -0.5 | 1.648721 | 2.648721 | 0.377541 | 0 | no |
| 5 | 2 | 0 | 0.5 | 0.606531 | 1.606531 | 0.622459 | 1 | no |
| 6 | 3 | 1 | 1.5 | 0.223130 | 1.223130 | 0.817574 | 1 | yes |
| 7 | 4 | 1 | 2.5 | 0.082085 | 1.082085 | 0.924142 | 1 | yes |
| 8 | 5 | 1 | 3.5 | 0.030197 | 1.030197 | 0.970688 | 1 | yes |

Six of the eight predictions match the labels, so `accuracy = 6 / 8 = 0.75`. This is a deterministic arithmetic fact about this synthetic eight-point example with these hand-picked coefficients, not evidence about real-world performance or generalization.

### Decision boundary, new inputs, and symmetry

With threshold `0.5`, the boundary occurs where `p = 0.5`, which means `σ(z) = 0.5`, which means `z = 0`. For this model:

```text
-1.5 + 1.0*x = 0
x* = 1.5
```

Check: `z(1.5) = 0` and `σ(0) = 0.5`. The boundary is not automatically at `x = 0`; it is wherever the model score equals zero.

New inputs:

| new `x` | `z` | `exp(-z)` | `1+exp(-z)` | `p = σ(z)` | `y_hat` | note |
|---:|---:|---:|---:|---:|---:|---|
| 1.5 | 0.0 | 1.000000 | 2.000000 | 0.500000 | 1 | exactly on the boundary; tie-break predicts class 1 |
| -4.0 | -5.5 | 244.691932 | 245.691932 | 0.004070 | 0 | arithmetic only; no extrapolation claim |

Symmetry checks: `σ(-0.5) + σ(0.5) = 0.377541 + 0.622459 = 1.000000` and `σ(-1.5) + σ(1.5) = 0.182426 + 0.817574 = 1.000000`.

## S6 — NumPy implementation: evaluating the fixed model

The hand calculation is the main idea. Now we reproduce it with NumPy. This is still evaluation, not training.

In [3]:
def sigmoid(z):
    """Logistic sigmoid: sigma(z) = 1 / (1 + exp(-z))."""
    return 1.0 / (1.0 + np.exp(-z))

z = b + X @ w
exp_neg_z = np.exp(-z)
denominator = 1.0 + exp_neg_z
p = sigmoid(z)
y_hat = (p >= 0.5).astype(int)
accuracy = np.mean(y_hat == y)

print("z:", z)
print("p:", p)
print("y_hat:", y_hat)
print("accuracy:", accuracy)

z: [-3.5 -2.5 -1.5 -0.5  0.5  1.5  2.5  3.5]
p: [0.029312 0.075858 0.182426 0.377541 0.622459 0.817574 0.924142 0.970688]
y_hat: [0 0 0 0 1 1 1 1]
accuracy: 0.75


In [4]:
correct = y_hat == y

print("i  x     y  z      exp(-z)     1+exp(-z)   p          y_hat  correct")
for i in range(X.shape[0]):
    print(
        f"{i + 1:<2} "
        f"{X[i, 0]:>4.1f} "
        f"{int(y[i]):>2} "
        f"{z[i]:>5.1f} "
        f"{exp_neg_z[i]:>11.6f} "
        f"{denominator[i]:>12.6f} "
        f"{p[i]:>10.6f} "
        f"{int(y_hat[i]):>6} "
        f"{bool(correct[i])}"
    )

i  x     y  z      exp(-z)     1+exp(-z)   p          y_hat  correct
1  -2.0  0  -3.5   33.115452    34.115452   0.029312      0 True
2  -1.0  0  -2.5   12.182494    13.182494   0.075858      0 True
3   0.0  0  -1.5    4.481689     5.481689   0.182426      0 True
4   1.0  1  -0.5    1.648721     2.648721   0.377541      0 False
5   2.0  0   0.5    0.606531     1.606531   0.622459      1 False
6   3.0  1   1.5    0.223130     1.223130   0.817574      1 True
7   4.0  1   2.5    0.082085     1.082085   0.924142      1 True
8   5.0  1   3.5    0.030197     1.030197   0.970688      1 True


## S7 — Verify the computation against frozen references

Displayed values round to six decimal places for readability. The assertions compare against full-precision references.

In [5]:
expected_z = np.array([-3.5, -2.5, -1.5, -0.5, 0.5, 1.5, 2.5, 3.5], dtype=np.float64)
expected_exp_neg_z = np.array([33.1154519586923, 12.1824939607035, 4.48168907033806, 1.64872127070013, 0.606530659712633, 0.22313016014843, 0.0820849986238988, 0.0301973834223185], dtype=np.float64)
expected_denominator = np.array([34.1154519586923, 13.1824939607035, 5.48168907033806, 2.64872127070013, 1.60653065971263, 1.22313016014843, 1.0820849986239, 1.03019738342232], dtype=np.float64)
expected_p = np.array([0.0293122307513563, 0.0758581800212435, 0.182425523806356, 0.377540668798145, 0.622459331201855, 0.817574476193644, 0.924141819978757, 0.970687769248644], dtype=np.float64)
expected_y_hat = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=int)

assert X.shape == (8, 1)
assert y.shape == (8,)
assert w.shape == (1,)
assert X.dtype == np.float64
assert y.dtype == np.float64
assert w.dtype == np.float64
np.testing.assert_array_equal(z, expected_z)
np.testing.assert_allclose(exp_neg_z, expected_exp_neg_z, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(denominator, expected_denominator, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(p, expected_p, rtol=RTOL, atol=ATOL)
np.testing.assert_array_equal(y_hat, expected_y_hat)
assert accuracy == 0.75
assert sigmoid(0.0) == 0.5
assert b + w[0] * 1.5 == 0.0

new_X = np.array([[1.5], [-4.0]], dtype=np.float64)
new_z = b + new_X @ w
new_p = sigmoid(new_z)
new_y_hat = (new_p >= 0.5).astype(int)
np.testing.assert_array_equal(new_z, np.array([0.0, -5.5], dtype=np.float64))
np.testing.assert_allclose(new_p, np.array([0.5, 0.004070137715896128], dtype=np.float64), rtol=RTOL, atol=ATOL)
np.testing.assert_array_equal(new_y_hat, np.array([1, 0], dtype=int))

print("PASS: fixed logistic-regression forward pass matches the frozen references.")

PASS: fixed logistic-regression forward pass matches the frozen references.


In [6]:
grid = np.arange(-6.0, 6.0 + 0.25, 0.25, dtype=np.float64)
grid_p = sigmoid(grid)
assert np.all((grid_p > 0.0) & (grid_p < 1.0))
assert np.all(np.diff(grid_p) > 0.0)
for value in [0.5, 1.5, 2.5, 3.5]:
    np.testing.assert_allclose(sigmoid(-value) + sigmoid(value), 1.0, rtol=RTOL, atol=ATOL)
print("PASS: sigmoid range, monotonicity, and symmetry checks hold on the lesson grid.")

PASS: sigmoid range, monotonicity, and symmetry checks hold on the lesson grid.


## S8 — Independent checks: same relationship, different implementation

The NumPy implementation is the primary code path. As an independent check, we can compute the same probabilities with pure Python’s `math.exp`. This cross-check does not call our `sigmoid` function.

In [7]:
pure_python_p = []
for row in X:
    z_i = b + w[0] * float(row[0])
    p_i = 1.0 / (1.0 + math.exp(-z_i))
    pure_python_p.append(p_i)

np.testing.assert_allclose(pure_python_p, expected_p, rtol=RTOL, atol=ATOL)
print("PASS: pure-Python math.exp agrees with the NumPy path.")

PASS: pure-Python math.exp agrees with the NumPy path.


A library estimator can also be used as a separate implementation check, but not as the derivation. In scikit-learn, `LogisticRegression` has a `decision_function` that returns linear scores and `predict_proba` that returns probabilities. The consistency relationship is `predict_proba(X)[:, 1] = sigmoid(decision_function(X))`. Because this lesson’s coefficients are hand-picked, a fitted scikit-learn model should **not** be expected to reproduce `b = -1.5` and `w = 1.0`. Training belongs to 03.02.

## S9 — Visualizing probabilities and the boundary

The same forward pass can be viewed in two spaces: score space and feature space.

### V1 — Sigmoid in score space

![Sigmoid curve in z-space](assets/logistic_sigmoid_z_space.png)

**Caption:** The sigmoid maps a linear score `z` to a probability `p = σ(z)`. The midpoint is `σ(0) = 0.5`, and the vertical dashed line marks the default decision boundary at `z = 0`. The eight frozen training examples sit on the curve; the two ringed points nearest the midpoint are the two incorrect hard classifications in this toy table. This synthetic teaching figure is not evidence of real-world performance.

**Text equivalent:** The plot shows the sigmoid function from `z = -6` to `z = 6`. The curve rises smoothly from near `0` to near `1`, passing through `(0, 0.5)`. Blue circles mark true class `0`; vermillion squares mark true class `1`; black rings mark the two misclassified points at `z = -0.5` and `z = 0.5`. Shaded regions indicate `predict 0` for `z < 0` and `predict 1` for `z >= 0`.

### V2 — Feature space: probability curve and decision boundary

![Feature-space logistic model and decision boundary](assets/logistic_feature_boundary.png)

**Caption:** The smooth curve is the predicted probability `p(x) = σ(-1.5 + x)`. The horizontal dotted line marks threshold `p = 0.5`; the vertical dashed line marks the decision boundary `x* = 1.5`. Training labels are plotted at heights `0` and `1`, while the curve shows probabilities. Points are synthetic, deterministic teaching data; the displayed accuracy is not evidence of real-world performance.

**Text equivalent:** The plot shows feature `x` on the horizontal axis and probability/class on the vertical axis. Class-0 labels are blue circles at height `0`, and class-1 labels are vermillion squares at height `1`. The logistic curve rises from near `0` on the left to near `1` on the right and crosses `p = 0.5` at `x = 1.5`. The misclassified points at `x = 1` and `x = 2` are ringed. New inputs are black diamonds at `x = 1.5` with `p = 0.5` and at `x = -4` with `p = 0.004`.

The curve is the probability. The shaded regions are class predictions after thresholding. This is why `p` and `y_hat` must be kept separate.

## S10 — Common mistakes to avoid

- Logistic regression is a classification model despite its name.
- `z` is not a probability. At `x = 3`, `z = 1.5` but `p = 0.817574`.
- `p` and `y_hat` are not the same. At `x = 2`, `p = 0.622459`, so `y_hat = 1`.
- The boundary is not always at `x = 0`; here it solves `-1.5 + x = 0`, so `x* = 1.5`.
- `0.5` is a common threshold, not the only valid threshold. Changing the threshold changes `y_hat`, not `b`, `w`, `z`, or `p`.
- Classes need not be perfectly separable. This table deliberately has `x = 1, y = 1` and `x = 2, y = 0`.
- The coefficients in this lesson are hand-picked. Training appears in 03.02.
- MSE is a poor default for sigmoid-output classification models; the classification loss is introduced in 03.02.

Numerical caveat: in exact mathematics, finite `z` gives `0 < σ(z) < 1`. In `float64`, extremely large scores can round to exactly `1.0` or `0.0`. With the direct formula used here, `σ(z)` rounds to exactly `1.0` around `z >= 36.7368`, and negative scores near `z <= -709.7827` overflow through `exp(-z)`. Our largest lesson score magnitude is `5.5`, far from these edges.

## S11 — Exercises

**E1 — Forward pass for one new point.** Using `b = -1.5`, `w = 1.0`, compute `z`, `exp(-z)`, `1 + exp(-z)`, `σ(z)`, and `y_hat` for `x = 0.5`.

Answer:

```text
z = -1.5 + 1.0*(0.5) = -1.0
exp(-z) = exp(1.0) = 2.718282
1 + exp(-z) = 3.718282
σ(z) = 1 / 3.718282 = 0.268941
y_hat = 0
```

**E2 — Negative coefficient and boundary.** A different model has `b = 2.0`, `w = -0.5`. Where is the boundary, which side predicts class `1`, and what is the prediction for `x = 8`?

Answer:

```text
boundary: 2.0 - 0.5*x = 0 → x* = 4.0
because w < 0, class 1 is predicted for x <= 4
for x = 8: z = -2.0
exp(-z) = 7.389056
1 + exp(-z) = 8.389056
σ(z) = 0.119203
y_hat = 0
```

**E3 — Two-feature transfer.** Use `z = -4 + x1 + x2`, `p = σ(z)`, and `y_hat = 1 if p >= 0.5 else 0`.

| i | `x1` | `x2` | `y` | `z` | `exp(-z)` | `1+exp(-z)` | `σ(z)` | `y_hat` | Correct? |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---:|
| 1 | 1 | 1 | 0 | -2 | 7.389056 | 8.389056 | 0.119203 | 0 | yes |
| 2 | 2 | 1 | 0 | -1 | 2.718282 | 3.718282 | 0.268941 | 0 | yes |
| 3 | 1 | 2 | 0 | -1 | 2.718282 | 3.718282 | 0.268941 | 0 | yes |
| 4 | 3 | 1 | 0 | 0 | 1.000000 | 2.000000 | 0.500000 | 1 | no |
| 5 | 3 | 3 | 1 | 2 | 0.135335 | 1.135335 | 0.880797 | 1 | yes |
| 6 | 4 | 3 | 1 | 3 | 0.049787 | 1.049787 | 0.952574 | 1 | yes |
| 7 | 3 | 4 | 1 | 3 | 0.049787 | 1.049787 | 0.952574 | 1 | yes |
| 8 | 4 | 4 | 1 | 4 | 0.018316 | 1.018316 | 0.982014 | 1 | yes |

The boundary is:

```text
-4 + x1 + x2 = 0
x1 + x2 = 4
```

The class-1 region is `x1 + x2 >= 4`. Accuracy is `7/8 = 0.875`. Row 4 lies exactly on the boundary: `z = 0`, `p = 0.5`, and the tie-break predicts class `1`, which is wrong for its label `0`.

**E4 — Can any one-threshold model classify the primary table perfectly?**

No. The primary table contains `x = 1, y = 1` and `x = 2, y = 0`. A one-feature monotone threshold rule cannot put `x = 1` on the class-1 side while putting the larger point `x = 2` on the class-0 side. At least one point must be wrong; the best possible accuracy for a one-threshold monotone rule on this toy table is `7/8`.


## S12 — Takeaways and what comes next

Logistic regression uses this pipeline:

```text
features → linear score z → sigmoid probability p → thresholded class y_hat
```

Main takeaways: binary classification uses labels such as `0` and `1`; logistic regression uses `z = b + wᵀx`; the sigmoid converts `z` into `p`; a threshold converts `p` into `y_hat`; with threshold `0.5`, the decision boundary occurs where `z = 0`; and this lesson evaluated hand-picked coefficients rather than training them.

The largest missing piece is deliberate: we supplied `b` and `w`. In 03.02, we will define a classification loss that scores a choice of coefficients and use gradient descent to choose coefficients instead of hand-picking them.

References: Cox (1958), Hastie/Tibshirani/Friedman *The Elements of Statistical Learning* Ch. 4, James et al. *An Introduction to Statistical Learning* classification chapter, Bishop *Pattern Recognition and Machine Learning* Ch. 4, Goodfellow/Bengio/Courville *Deep Learning* §6.2.2.2, NumPy `exp` documentation, and scikit-learn 1.5 `LogisticRegression` documentation.